# AI Video Generation API - Colab
Run CogVideoX-2B on free T4 GPU, exposes Gradio API for local app

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q gradio diffusers transformers accelerate sentencepiece protobuf imageio[ffmpeg]


In [ ]:
import torch, gradio as gr, os, time, random
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

MODEL = "THUDM/CogVideoX-2B"
print(f"Loading {MODEL}...")

pipe = CogVideoXPipeline.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print("Model loaded!")

def generate(prompt, steps=25, guidance=6.0, seed=-1):
    if seed < 0:
        seed = random.randint(0, 2**31)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    print(f"Generating: {prompt}")
    t0 = time.time()
    result = pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        num_frames=49
    ).frames[0]
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.1f}s")
    path = export_to_video(result, fps=8)
    return path

iface = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", value="A tiger walking through tall grass at sunrise, cinematic"),
        gr.Slider(5, 50, 25, label="Steps"),
        gr.Slider(1.0, 10.0, 6.0, label="Guidance"),
        gr.Number(-1, label="Seed (-1=random)", precision=0),
    ],
    outputs=gr.Video(label="Video"),
    title="AI Video Generator",
)

print("="*50)
print("COPY THE GRADIO LINK BELOW (xxx.gradio.app)")
print("Set it as T2V_API_URL in your .env file")
print("="*50)
iface.launch(share=True)